# Module 2: `nn.py` — Matrix Multiplication VJP (`@`), `Linear` (He Init) & `MLP`

This notebook demonstrates matrix multiplication Vector-Jacobian Products ($G Y^T$ and $X^T G$), shape transformations (`reshape`, `transpose`), He/Kaiming weight initialization, and multi-layer perceptron (`MLP`) forward/backward passes.

## 1. Imports & Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import numpy as np
from tensorgrad.tensor import Tensor, numerical_grad_check
from tensorgrad.nn import Linear, MLP
print('Loaded tensorgrad.nn successfully.')

Loaded tensorgrad.nn successfully.


## 2. Matrix Multiplication VJP (`Z = X @ Y`)
For $X \in \mathbb{R}^{N \times D}$ and $Y \in \mathbb{R}^{D \times M}$, $\frac{\partial L}{\partial X} = G Y^T$ and $\frac{\partial L}{\partial Y} = X^T G$.

In [2]:
rng = np.random.default_rng(13)
X = Tensor(rng.normal(size=(6, 4)))
Y = Tensor(rng.normal(size=(4, 5)))
Z = X @ Y
Z.backward(grad=np.ones_like(Z.data))
print(f'X.shape={X.shape}, Y.shape={Y.shape} -> Z.shape={Z.shape}')
print(f'X.grad.shape={X.grad.shape}, Y.grad.shape={Y.grad.shape}')
print('X.grad[:2]:\n', np.round(X.grad[:2], 4))

X.shape=(6, 4), Y.shape=(4, 5) -> Z.shape=(6, 5)
X.grad.shape=(6, 4), Y.grad.shape=(4, 5)
X.grad[:2]:
 [[ 0.6271  0.459   0.091  -1.2983]
 [ 0.6271  0.459   0.091  -1.2983]]


## 3. He Initialization & Multi-Layer Perceptron (`MLP`)
We construct a 3-layer `MLP(4, [16, 8], 2)` and verify end-to-end backpropagation with `numerical_grad_check`.

In [3]:
mlp = MLP(in_dim=4, hidden_dims=[16, 8], out_dim=2, activation='relu', rng=rng)
print('MLP architecture:', mlp)
print('Parameter shapes:', [p.shape for p in mlp.parameters()])

X_batch = Tensor(rng.normal(size=(5, 4)))
head_w = rng.normal(size=(5, 2))
chk = numerical_grad_check(lambda t: (mlp(t) * head_w).sum(), X_batch)
print(f"End-to-end MLP Gradcheck passed: {chk['passed']} | rel_error={chk['rel_error']:.3e}")

MLP architecture: MLP(Sequential(Linear(in_features=4, out_features=16, bias=True), ReLU(), Linear(in_features=16, out_features=8, bias=True), ReLU(), Linear(in_features=8, out_features=2, bias=True)))
Parameter shapes: [(4, 16), (16,), (16, 8), (8,), (8, 2), (2,)]
End-to-end MLP Gradcheck passed: True | rel_error=8.767e-11


## Conclusion
`nn.py` provides analytical matrix VJPs, shape operators, He-initialized linear layers, and modular `MLP` containers ready for gradient-based optimization.